# v01 — Baseline

Primeira submissão. Objetivo: ter um pipeline válido e reproduzível de ponta a ponta.

- **Baseline 1:** frequência histórica das classes (ajustada só no treino de cada fold).
- **Baseline 2:** regressão logística multinomial com as 17 variáveis numéricas pré-jogo, `C=1`.

Validação: *walk-forward* por temporada (treina em tudo antes de `s`, valida em `s`), com `s` de 2016 a 2021. Odds não entram no modelo.

In [1]:
import warnings
from datetime import date
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.compose import ColumnTransformer
from sklearn.dummy import DummyClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)

DATA = Path('../data')
SUB_DIR = Path('../submissions')
SUB_DIR.mkdir(exist_ok=True)
LOG_PATH = Path('../experimentos.csv')

CLASS_ORDER = ['A', 'D', 'H']
RANDOM_STATE = 42
FOLDS = [2016, 2017, 2018, 2019, 2020, 2021]  # walk-forward: treino = temporadas anteriores

train = pd.read_csv(DATA / 'train.csv')
test = pd.read_csv(DATA / 'test.csv')
print(train.shape, test.shape)

(3419, 26) (1725, 22)


In [2]:
NUM_FEATURES = ['elo_home', 'elo_away', 'elo_diff', 'form_pts_home', 'form_pts_away',
                'form_gf_home', 'form_ga_home', 'form_gf_away', 'form_ga_away',
                'home_form_as_host', 'away_form_as_visitor', 'rest_days_home', 'rest_days_away',
                'season_ppg_home', 'season_ppg_away', 'round_n', 'h2h_home_pts']


def add_features(df):
    """Features derivadas usando apenas colunas da propria linha."""
    out = df.copy()
    out['venue_form_diff'] = out['home_form_as_host'] - out['away_form_as_visitor']
    out['form_pts_diff'] = out['form_pts_home'] - out['form_pts_away']
    out['gd_diff'] = (out['form_gf_home'] - out['form_ga_home']) - (out['form_gf_away'] - out['form_ga_away'])
    return out


train = add_features(train)
test = add_features(test)

In [3]:
def make_logreg(num_cols, cat_cols=(), C=1.0):
    transformers = [('num', Pipeline([('impute', SimpleImputer(strategy='median')),
                                      ('scale', StandardScaler())]), list(num_cols))]
    if cat_cols:
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore'), list(cat_cols)))
    return Pipeline([('prep', ColumnTransformer(transformers)),
                     ('model', LogisticRegression(C=C, max_iter=5000, random_state=RANDOM_STATE))])


def predict_ordered(model, X):
    """Alinha predict_proba a ['A', 'D', 'H'] pelas classes do modelo."""
    raw = pd.DataFrame(model.predict_proba(X), columns=model.named_steps['model'].classes_)
    return raw.reindex(columns=CLASS_ORDER, fill_value=0.0).values


def walk_forward(build, cols):
    """Previsoes out-of-fold por temporada; o modelo e reconstruido do zero a cada fold."""
    preds, ys = {}, {}
    for season in FOLDS:
        tr = train[train['Season'] < season]
        va = train[train['Season'] == season]
        model = build()
        model.fit(tr[cols], tr['Res'])
        preds[season] = predict_ordered(model, va[cols])
        ys[season] = va['Res'].values
    return preds, ys


def summarize(preds, ys, name):
    rows = [{'Season': s, 'n': len(ys[s]), 'LogLoss': log_loss(ys[s], preds[s], labels=CLASS_ORDER)}
            for s in FOLDS]
    df = pd.DataFrame(rows)
    mean = np.average(df['LogLoss'], weights=df['n'])
    print(f'{name}: media ponderada = {mean:.4f} | desvio entre folds = {df.LogLoss.std():.4f}')
    return df, mean

In [4]:
def save_submission(proba, path):
    sub = pd.DataFrame(proba, columns=CLASS_ORDER)
    sub.insert(0, 'Id', test['Id'].values)
    assert list(sub.columns) == ['Id', 'A', 'D', 'H']
    assert len(sub) == len(test) and (sub['Id'].values == test['Id'].values).all()
    vals = sub[CLASS_ORDER].values
    assert np.isfinite(vals).all() and (vals > 0).all()
    assert np.allclose(vals.sum(axis=1), 1.0, atol=1e-9)
    sub.to_csv(path, index=False)
    print('salvo em', path, sub.shape)
    return sub


def log_experiment(version, change, features, model_desc, params, folds_df, mean, decision, csv_path):
    row = {'versao': version, 'data': date.today().isoformat(), 'mudanca': change,
           'features': features, 'modelo': model_desc, 'parametros': params,
           'folds': ','.join(map(str, FOLDS)),
           'logloss_por_fold': ';'.join(f'{s}:{v:.4f}' for s, v in zip(folds_df.Season, folds_df.LogLoss)),
           'logloss_medio': round(mean, 4), 'desvio': round(folds_df.LogLoss.std(), 4),
           'decisao': decision, 'csv': csv_path}
    log = pd.read_csv(LOG_PATH) if LOG_PATH.exists() else pd.DataFrame(columns=list(row.keys()))
    log = log[log['versao'] != version]
    log = pd.concat([log, pd.DataFrame([row])], ignore_index=True)
    log.to_csv(LOG_PATH, index=False)
    return log

## Inspeção rápida

In [5]:
print(train['Res'].value_counts(normalize=True).round(3))
print(train.groupby('Season')['Res'].value_counts(normalize=True).unstack().round(3))
print('ausentes (treino):')
print(train[NUM_FEATURES].isna().mean().round(3).sort_values(ascending=False).head(5))

Res
H    0.492
D    0.268
A    0.240
Name: proportion, dtype: float64
Res         A      D      H
Season                     
2013    0.232  0.284  0.484
2014    0.239  0.242  0.518
2015    0.234  0.239  0.526
2016    0.219  0.248  0.533
2017    0.289  0.271  0.439
2018    0.179  0.289  0.532
2019    0.258  0.258  0.484
2020    0.266  0.284  0.450
2021    0.245  0.297  0.458
ausentes (treino):
h2h_home_pts            0.192
season_ppg_away         0.026
season_ppg_home         0.026
away_form_as_visitor    0.005
home_form_as_host       0.005
dtype: float64


## Validação temporal

In [6]:
prior_preds, ys = walk_forward(lambda: Pipeline([('model', DummyClassifier(strategy='prior'))]), ['elo_diff'])
prior_df, prior_mean = summarize(prior_preds, ys, 'frequencia historica')

lr_preds, _ = walk_forward(lambda: make_logreg(NUM_FEATURES, C=1.0), NUM_FEATURES)
lr_df, lr_mean = summarize(lr_preds, ys, 'logreg 17 numericas, C=1')

pd.DataFrame({'freq': prior_df.LogLoss.values, 'logreg': lr_df.LogLoss.values}, index=FOLDS).round(4)

frequencia historica: media ponderada = 1.0517 | desvio entre folds = 0.0307


logreg 17 numericas, C=1: media ponderada = 1.0337 | desvio entre folds = 0.0455


,freq,logreg
2016,1.0150,1.0119
2017,1.0874,1.1087
2018,1.0148,0.9857
2019,1.0520,0.9977
2020,1.0747,1.0588
2021,1.0663,1.0395


A regressão logística fica abaixo da frequência histórica na média; vira o campeão inicial.

## Treino final e submissão

In [7]:
final = make_logreg(NUM_FEATURES, C=1.0).fit(train[NUM_FEATURES], train['Res'])
proba = predict_ordered(final, test[NUM_FEATURES])
sub = save_submission(proba, SUB_DIR / 'submission_v01.csv')
log_experiment('v01', 'baseline logreg', '17 numericas', 'LogisticRegression', 'C=1',
               lr_df, lr_mean, 'campeao inicial', 'submissions/submission_v01.csv')
sub.head()

salvo em ..\submissions\submission_v01.csv (1725, 4)


,Id,A,D,H
0,3419,0.209081,0.279841,0.511077
1,3420,0.317412,0.311746,0.370842
2,3421,0.183478,0.230482,0.586040
3,3422,0.254579,0.185412,0.560008
4,3423,0.120333,0.182468,0.697199
